In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
paths = []

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # print(os.path.join(dirname, filename))
        paths.append(os.path.join(dirname, filename))

# You can write up to 20GB to t2he current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Шум в разметке

In [ ]:
from pathlib import Path
import json

root = Path.cwd()
out = root / 'output'
out.mkdir(exist_ok=True)

df = pd.read_csv(root / 'train.csv')
oof = pd.read_csv(root / 'oof.csv')
probs = np.load(root / 'oof_probs.npy')
classes = json.loads((root / 'classes.json').read_text(encoding='utf-8'))

df['label'] = df['Category'].fillna('NA') + ':' + df['Misconception'].fillna('NA')
df['text_clean'] = df['StudentExplanation'].fillna('').astype(str).str.lower().str.strip()

assert len(df) == len(oof) == len(probs)
assert np.array_equal(df['row_id'].to_numpy(), oof['row_id'].to_numpy())
assert (df['label'].to_numpy() == oof['target'].to_numpy()).all()
assert probs.shape[1] == len(classes)
assert np.isfinite(probs).all()
assert np.allclose(probs.sum(axis=1), 1, atol=1e-4)

print(len(df), probs.shape, len(classes))

In [ ]:
keys = ['QuestionId', 'MC_Answer', 'text_clean']
groups = df.groupby(keys, dropna=False)['label'].nunique()
conflicts = groups[groups > 1]
conflict_keys = conflicts.rename('n_labels').reset_index()[keys]
exact_conflicts = df.merge(conflict_keys, on=keys, how='inner')
exact_conflicts.to_csv(out / 'exact_conflicts.csv', index=False)
print('Групп:', len(conflicts))
print('Строк:', len(exact_conflicts))
display(exact_conflicts.head(20))

In [ ]:
from sklearn.neighbors import NearestNeighbors

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=2, dtype=np.float32)
X_text = vectorizer.fit_transform(df['text_clean'])
near_pairs = []

for _, group in df.groupby(['QuestionId', 'MC_Answer'], dropna=False):
    ids = group.index.to_numpy()
    if len(ids) < 2:
        continue
    nn = NearestNeighbors(n_neighbors=min(10, len(ids)), metric='cosine', algorithm='brute', n_jobs=1)
    nn.fit(X_text[ids])
    distances, neighbors = nn.kneighbors(X_text[ids])
    for i, source in enumerate(ids):
        for distance, neighbor in zip(distances[i], neighbors[i]):
            target = ids[neighbor]
            similarity = 1 - distance
            if source >= target or similarity < 0.9 or similarity >= 0.999999:
                continue
            if df.at[source, 'label'] == df.at[target, 'label']:
                continue
            near_pairs.append((df.at[source, 'row_id'], df.at[target, 'row_id'], similarity,
                               df.at[source, 'StudentExplanation'], df.at[target, 'StudentExplanation'],
                               df.at[source, 'label'], df.at[target, 'label']))

near_conflicts = pd.DataFrame(near_pairs, columns=['row_id_1', 'row_id_2', 'similarity', 'text_1', 'text_2', 'label_1', 'label_2'])
near_conflicts.to_csv(out / 'near_conflicts.csv', index=False)
print('Почти-дублей с разными метками:', len(near_conflicts))
display(near_conflicts.head(20))


In [ ]:
class_to_idx = {label: i for i, label in enumerate(classes)}
y = df['label'].map(class_to_idx)
assert y.notna().all()
y = y.to_numpy(dtype=int)
assert np.array_equal(y, oof['label_idx'].to_numpy())

counts = np.bincount(y, minlength=len(classes))
eligible = np.flatnonzero(counts >= 5)
mask = np.isin(y, eligible)
row_positions = np.flatnonzero(mask)
labels_cl = np.searchsorted(eligible, y[mask])
probs_cl = probs[mask][:, eligible].copy()
row_sums = probs_cl.sum(axis=1, keepdims=True)
assert np.all(row_sums > 0)
probs_cl /= row_sums

print('Всего строк:', len(df))
print('Для Cleanlab:', len(labels_cl))
print('Отложено:', (~mask).sum())


In [ ]:
from cleanlab.filter import find_label_issues

issues = find_label_issues(
    labels=labels_cl,
    pred_probs=probs_cl,
    return_indices_ranked_by='self_confidence',
    n_jobs=1
)

print('Cleanlab нашёл:', len(issues))
print('Доля среди проверяемых классов:', f'{len(issues) / len(labels_cl):.2%}')


In [ ]:
positions = row_positions[issues]
suspicious = df.iloc[positions].copy()
suspicious['model_label'] = np.asarray(classes)[eligible[probs_cl[issues].argmax(axis=1)]]
suspicious['p_given_label'] = probs_cl[issues, labels_cl[issues]]
suspicious['p_model_label'] = probs_cl[issues].max(axis=1)
suspicious['fold'] = oof.iloc[positions]['fold'].to_numpy()
suspicious['cleanlab_rank'] = np.arange(1, len(suspicious) + 1)
suspicious['verdict'] = ''
suspicious['comment'] = ''

columns = ['cleanlab_rank', 'row_id', 'fold', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation', 'label', 'model_label', 'p_given_label', 'p_model_label', 'verdict', 'comment']
suspicious = suspicious[columns]
suspicious.to_csv(out / 'cleanlab_suspicious.csv', index=False)
suspicious.head(50).to_csv(out / 'manual_review_50.csv', index=False)
suspicious.groupby(suspicious['label'].str.split(':').str[0]).size().sort_values(ascending=False).to_csv(out / 'suspects_by_category.csv', header=['count'])

display(suspicious.head(20))
print('Файлы сохранены в', out)
